# Passo 3: definição do alvo

**Alvo = 1 ("alta relevante")** quando, 6 meses depois, a taxa de inadimplência do recorte estiver ao mesmo tempo:
- mais de **0,5 ponto percentual** acima da taxa atual, **e**
- mais de **20%** acima da taxa atual.

Caso contrário, alvo = 0. Nos últimos 6 meses de cada recorte o futuro ainda não é conhecido, então o alvo fica vazio (`<NA>`).

In [5]:
import pandas as pd

from src.dados import PASTA_PROCESSED, CHAVES_RECORTE

# Parâmetros da definição do alvo
HORIZONTE = 6        # meses à frente
ALTA_MIN_PP = 0.5    # alta mínima em pontos percentuais
ALTA_MIN_REL = 0.20  # alta mínima relativa (20%)

# Painel do passo 2: 386 recortes x 48 meses
painel = pd.read_parquet(PASTA_PROCESSED / "painel_filtrado.parquet")

# Ordena por recorte e data: o deslocamento abaixo depende dessa ordem
painel = painel.sort_values(CHAVES_RECORTE + ["data_base"]).reset_index(drop=True)
print(painel.shape)

(18528, 11)


In [6]:
# Taxa de inadimplência do mês, em %
painel["taxa_inad"] = 100 * painel["carteira_inadimplencia"] / painel["carteira_ativa"]

# Taxa 6 meses à frente: shift(-6) "puxa" o valor de 6 linhas abaixo,
# dentro de cada recorte (groupby), sem misturar recortes diferentes.
# Funciona porque todo recorte tem os 48 meses seguidos, sem buracos.
painel["taxa_inad_futura"] = (
    painel.groupby(CHAVES_RECORTE, observed=True)["taxa_inad"].shift(-HORIZONTE)
)

# As duas condições da alta relevante
subiu_pp = (painel["taxa_inad_futura"] - painel["taxa_inad"]) > ALTA_MIN_PP
# Multiplicar em vez de dividir evita divisão por zero quando a taxa atual é 0
subiu_rel = painel["taxa_inad_futura"] > painel["taxa_inad"] * (1 + ALTA_MIN_REL)

# Alvo 0/1; "Int8" é um inteiro que aceita valor vazio (<NA>)
painel["alvo"] = (subiu_pp & subiu_rel).astype("Int8")
painel.loc[painel["taxa_inad_futura"].isna(), "alvo"] = pd.NA

print(painel["alvo"].value_counts(dropna=False))

alvo
0       13548
1        2664
<NA>     2316
Name: count, dtype: Int64


In [7]:
# Conferência: proporção de alvo = 1 (as linhas <NA> ficam de fora da média)
com_alvo = painel.dropna(subset=["alvo"])
print(f"Linhas com alvo: {len(com_alvo)}")
print(f"Alvo = 1: {100 * com_alvo['alvo'].mean():.1f}%\n")

print("Alvo = 1 (%) por ano do mês-base:")
print((100 * com_alvo.groupby(com_alvo["data_base"].dt.year)["alvo"].mean()).round(1))

# Guarda o painel inteiro (inclusive as linhas <NA>, úteis para calcular variáveis no passo 4)
painel.to_parquet(PASTA_PROCESSED / "painel_alvo.parquet", index=False)

Linhas com alvo: 16212
Alvo = 1: 16.4%

Alvo = 1 (%) por ano do mês-base:
data_base
2022    16.0
2023    11.0
2024    18.1
2025    24.9
Name: alvo, dtype: Float64
